# Self-Correction and Iterative Refinement

LLMs make mistakes. They hallucinate facts, write buggy code, and miss edge cases. Self-correction teaches models to catch and fix their own errors through iterative refinement.

The core pattern: **Generate → Critique → Refine → Repeat**

This notebook covers:
- Critic-generator separation
- Self-reflection prompts
- Iterative refinement with stopping criteria
- When self-correction helps vs hurts

In [ ]:
import os
import json
from getpass import getpass
from dataclasses import dataclass
from openai import OpenAI
from pydantic import BaseModel, Field

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-4o-mini"

## Part 1: Basic Self-Reflection

The simplest form: ask the model to check its own work.

In [ ]:
def generate_with_reflection(prompt: str) -> dict:
    """Generate a response, then reflect and improve it."""
    
    # Step 1: Initial generation
    initial_response = client.responses.create(
        model=MODEL,
        input=prompt
    )
    initial = initial_response.output_text
    
    # Step 2: Self-reflection
    reflection_prompt = f"""Original task: {prompt}

Your response:
{initial}

Now critically review your response:
1. What mistakes or errors might it contain?
2. What important points did you miss?
3. How could the response be clearer or more accurate?

List the issues you found:"""
    
    reflection_response = client.responses.create(
        model=MODEL,
        input=reflection_prompt
    )
    reflection = reflection_response.output_text
    
    # Step 3: Refined response
    refine_prompt = f"""Original task: {prompt}

Your initial response:
{initial}

Issues identified:
{reflection}

Now provide an improved response that addresses these issues:"""
    
    refined_response = client.responses.create(
        model=MODEL,
        input=refine_prompt
    )
    
    return {
        "initial": initial,
        "reflection": reflection,
        "refined": refined_response.output_text
    }

# Test
result = generate_with_reflection(
    "Explain how a binary search algorithm works. Include the time complexity."
)

print("=== INITIAL ===")
print(result["initial"][:500])
print("\n=== REFLECTION ===")
print(result["reflection"][:500])
print("\n=== REFINED ===")
print(result["refined"][:500])

## Part 2: Critic-Generator Pattern

Separating the critic from the generator often works better than self-reflection. The critic has a different "mindset" focused on finding problems.

In [ ]:
class Critique(BaseModel):
    has_errors: bool = Field(description="Whether errors were found")
    errors: list[str] = Field(description="List of specific errors found")
    suggestions: list[str] = Field(description="Suggestions for improvement")
    score: int = Field(description="Quality score 1-10", ge=1, le=10)

def critic_generator_loop(
    task: str,
    max_iterations: int = 3,
    target_score: int = 8
) -> dict:
    """Generate, critique, and refine until quality threshold is met."""
    
    history = []
    
    # Initial generation
    response = client.responses.create(
        model=MODEL,
        input=task
    )
    current = response.output_text
    
    for iteration in range(max_iterations):
        # Critic evaluates
        schema = Critique.model_json_schema()
        schema["additionalProperties"] = False
        
        critic_response = client.responses.create(
            model=MODEL,
            instructions="You are a strict critic. Find every possible issue. Be thorough.",
            input=f"""Task: {task}

Response to evaluate:
{current}

Evaluate this response critically.""",
            text={
                "format": {
                    "type": "json_schema",
                    "name": "Critique",
                    "strict": True,
                    "schema": schema
                }
            }
        )
        
        critique = Critique.model_validate_json(critic_response.output_text)
        
        history.append({
            "iteration": iteration + 1,
            "response": current,
            "score": critique.score,
            "errors": critique.errors
        })
        
        # Check if we've reached target quality
        if critique.score >= target_score or not critique.has_errors:
            break
        
        # Generator improves based on critique
        improvement_prompt = f"""Original task: {task}

Your previous response:
{current}

Errors found:
{chr(10).join('- ' + e for e in critique.errors)}

Suggestions:
{chr(10).join('- ' + s for s in critique.suggestions)}

Provide an improved response that fixes these issues:"""
        
        improved = client.responses.create(
            model=MODEL,
            input=improvement_prompt
        )
        current = improved.output_text
    
    return {
        "final": current,
        "iterations": len(history),
        "history": history
    }

# Test
result = critic_generator_loop(
    "Write a Python function to find all prime numbers up to n using the Sieve of Eratosthenes.",
    max_iterations=3,
    target_score=8
)

print(f"Completed in {result['iterations']} iterations\n")
for h in result['history']:
    print(f"Iteration {h['iteration']}: Score {h['score']}/10")
    if h['errors']:
        print(f"  Errors: {h['errors'][:2]}")

print("\n=== FINAL RESPONSE ===")
print(result['final'])

## Part 3: Self-Debugging Code

For code generation, we can actually run the code and use errors as feedback.

In [ ]:
import traceback
import io
import sys

def execute_code(code: str) -> dict:
    """Execute Python code and capture output/errors."""
    # Capture stdout
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    
    result = {"success": False, "output": "", "error": ""}
    
    try:
        exec(code, {})
        result["success"] = True
        result["output"] = sys.stdout.getvalue()
    except Exception as e:
        result["error"] = f"{type(e).__name__}: {str(e)}\n{traceback.format_exc()}"
    finally:
        sys.stdout = old_stdout
    
    return result

def self_debugging_code_gen(
    task: str,
    max_attempts: int = 3
) -> dict:
    """Generate code, test it, debug based on actual errors."""
    
    attempts = []
    
    # Initial code generation
    response = client.responses.create(
        model=MODEL,
        input=f"""{task}

Write Python code that:
1. Implements the solution
2. Includes a test/demo at the end that prints results

Return ONLY the Python code, no markdown or explanations."""
    )
    
    code = response.output_text
    # Clean up markdown if present
    if "```python" in code:
        code = code.split("```python")[1].split("```")[0]
    elif "```" in code:
        code = code.split("```")[1].split("```")[0]
    
    for attempt in range(max_attempts):
        # Try to execute
        result = execute_code(code)
        
        attempts.append({
            "attempt": attempt + 1,
            "code": code,
            "success": result["success"],
            "output": result["output"],
            "error": result["error"]
        })
        
        if result["success"]:
            break
        
        # Debug based on error
        debug_prompt = f"""Task: {task}

Your code:
```python
{code}
```

Error when running:
{result['error']}

Fix the error. Return ONLY the corrected Python code, no markdown or explanations."""
        
        fixed = client.responses.create(
            model=MODEL,
            input=debug_prompt
        )
        code = fixed.output_text
        if "```python" in code:
            code = code.split("```python")[1].split("```")[0]
        elif "```" in code:
            code = code.split("```")[1].split("```")[0]
    
    return {
        "final_code": code,
        "success": attempts[-1]["success"],
        "attempts": attempts
    }

# Test with a task that often has bugs
result = self_debugging_code_gen(
    "Write a function that finds the longest palindromic substring in a given string. Test it with 'babad' and 'cbbd'."
)

print(f"Success: {result['success']}")
print(f"Attempts: {len(result['attempts'])}")

for a in result['attempts']:
    status = "✓" if a['success'] else "✗"
    print(f"\nAttempt {a['attempt']} {status}")
    if a['error']:
        print(f"Error: {a['error'][:200]}")
    if a['output']:
        print(f"Output: {a['output']}")

## Part 4: Math Problem Self-Verification

For math, we can verify answers by working backwards or using alternative methods.

In [ ]:
class MathSolution(BaseModel):
    reasoning: str = Field(description="Step-by-step reasoning")
    answer: str = Field(description="The final numerical answer")
    verification: str = Field(description="Verification by working backwards or alternative method")
    confident: bool = Field(description="Whether the verification confirms the answer")

def solve_with_verification(problem: str) -> dict:
    """Solve a math problem with built-in verification."""
    
    schema = MathSolution.model_json_schema()
    schema["additionalProperties"] = False
    
    response = client.responses.create(
        model=MODEL,
        input=f"""Solve this math problem step by step.

Problem: {problem}

After solving, verify your answer by:
1. Substituting your answer back into the original problem, OR
2. Solving it a different way

If verification fails, reconsider your solution.""",
        text={
            "format": {
                "type": "json_schema",
                "name": "MathSolution",
                "strict": True,
                "schema": schema
            }
        }
    )
    
    solution = MathSolution.model_validate_json(response.output_text)
    
    # If not confident, try again with explicit instruction to reconsider
    if not solution.confident:
        retry_response = client.responses.create(
            model=MODEL,
            input=f"""Problem: {problem}

Your previous attempt:
Reasoning: {solution.reasoning}
Answer: {solution.answer}
Verification failed: {solution.verification}

The verification showed an error. Carefully reconsider and solve again from scratch.""",
            text={
                "format": {
                    "type": "json_schema",
                    "name": "MathSolution",
                    "strict": True,
                    "schema": schema
                }
            }
        )
        solution = MathSolution.model_validate_json(retry_response.output_text)
    
    return {
        "problem": problem,
        "answer": solution.answer,
        "reasoning": solution.reasoning,
        "verification": solution.verification,
        "confident": solution.confident
    }

# Test
problems = [
    "If 3x + 7 = 22, what is x?",
    "A train travels 120 miles in 2 hours. How long to travel 300 miles at the same speed?",
    "What is 15% of 240?"
]

for problem in problems:
    result = solve_with_verification(problem)
    status = "✓" if result['confident'] else "?"
    print(f"{status} Problem: {problem}")
    print(f"  Answer: {result['answer']}")
    print(f"  Verified: {result['confident']}")
    print()

## Part 5: When Self-Correction Hurts

Self-correction isn't always beneficial. It can:
1. **Over-correct**: Change correct answers to wrong ones
2. **Increase hallucination**: The model "doubles down" on errors
3. **Add unnecessary complexity**: Simple answers become convoluted

### Cases where it often hurts:

In [ ]:
def compare_with_without_correction(prompt: str) -> dict:
    """Compare direct answer vs self-corrected answer."""
    
    # Direct answer
    direct = client.responses.create(
        model=MODEL,
        input=prompt
    ).output_text
    
    # With self-correction
    corrected_result = generate_with_reflection(prompt)
    
    return {
        "direct": direct,
        "corrected": corrected_result["refined"],
        "reflection": corrected_result["reflection"]
    }

# Test on a factual question (often over-corrects)
result = compare_with_without_correction(
    "What is the capital of Australia?"
)

print("=== FACTUAL QUESTION ===")
print(f"Direct: {result['direct'][:200]}")
print(f"\nCorrected: {result['corrected'][:200]}")
print(f"\nReflection found: {result['reflection'][:200]}")

## Guidelines for Self-Correction

| Task Type | Self-Correction? | Why |
|-----------|-----------------|-----|
| Code generation | ✓ Yes | Can test and debug with real errors |
| Math problems | ✓ Yes | Can verify by substitution |
| Creative writing | ✓ Yes | Refinement improves quality |
| Simple factual Q&A | ✗ No | Often over-corrects correct answers |
| Classification | ✗ No | Model tends to second-guess itself |
| Summarization | ~ Maybe | Can help catch omissions, but may add hallucinations |

**Key insight**: Self-correction works best when there's an objective way to verify correctness.

---

## Exercises

### Exercise 1: Multi-Stage Writing Refinement

Create a writing assistant that refines text in stages: grammar → clarity → style → conciseness.

In [ ]:
def multi_stage_writing_refinement(text: str) -> dict:
    """Refine writing through multiple specialized passes."""
    # Your implementation here
    pass

### Exercise 2: Confidence-Gated Correction

Only apply self-correction when the model's confidence is below a threshold.

In [ ]:
def confidence_gated_response(prompt: str, confidence_threshold: float = 0.7) -> dict:
    """Only self-correct if initial confidence is low."""
    # Your implementation here
    pass

### Exercise 3: Test-Driven Code Generation

Generate code by first writing tests, then generating code to pass them, then debugging.

In [ ]:
def test_driven_code_gen(specification: str) -> dict:
    """Generate tests first, then code to pass them."""
    # Your implementation here
    pass

---

## Summary

- **Self-reflection**: Model critiques its own output
- **Critic-generator**: Separate agents for generation and critique
- **Self-debugging**: Use actual execution errors as feedback
- **Verification**: Check answers by working backwards
- **Not always better**: Self-correction can over-correct or hallucinate

Use self-correction when you have an objective way to verify correctness.